In [0]:
%python
%pip install "h3>=3.7,<4.0" pygeohash shapely

In [0]:
dbutils.library.restartPython()

In [0]:
%python
import h3
import pygeohash as pgh
from pyspark.sql.functions import (
    col, lit, udf, monotonically_increasing_id, explode,  trim, upper, sha2, min as spark_min, max as spark_max,
    concat, lpad, when, current_date, cast, regexp_replace, round as spark_round, sequence, coalesce, lag, row_number
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, max as spark_max

#  UDFs for H3 
@udf(StringType())
def to_h3(lat, lon):
    try:
        return h3.geo_to_h3(float(lat), float(lon), resolution=9)
    except:
        return None

def to_num(c):
    return when(
        regexp_replace(col(c), "[^0-9.]", "") == "", lit(None)
    ).otherwise(
        regexp_replace(col(c), "[^0-9.]", "").cast("double")
    )

def to_int(c):
    return to_num(c).cast("int")

def to_stories(c):
    return spark_round(to_num(c), 0).cast("int")

In [0]:
# Read Bronze tables 
pluto = spark.table("cre_catalog.bronze.mappluto_raw")
tax   = spark.table("cre_catalog.bronze.nyc_tax_raw")

In [0]:
# Get latest tax record per PARID 
# Get only most current record per parcel
window = Window.partitionBy("PARID").orderBy(
    col("YEAR").desc(), 
    col("PERIOD").desc()
)

tax_latest = tax \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"Latest tax records: {tax_latest.count()}")

In [0]:
#  Join Pluto + Tax 
joined = pluto.join(
    tax_latest,
    pluto["BBL"].cast("string") == tax_latest["PARID"],
    "left"
)

#  Build dim_site 
dim_site = joined.select(

    # Site ID = PARID (BBL)
    pluto["BBL"].cast("string").alias("site_id"),

    # Location
    col("PARID").alias("parid"),
    pluto["Borough"].alias("boro"),
    pluto["Block"].cast("string").alias("block"),
    pluto["Lot"].cast("string").alias("lot"),
    tax_latest["HOUSENUM_LO"].alias("house_num_lo"),
    tax_latest["HOUSENUM_HI"].alias("house_num_hi"),
    tax_latest["STREET_NAME"].alias("street_name"),
    tax_latest["ZIP_CODE"].alias("zip_code"),
    tax_latest["ZONING"].alias("zoning"),
    tax_latest["CORNER"].alias("corner"),

    # Land
    tax_latest["LAND_AREA"].alias("land_area"),
    pluto["LotArea"].alias("lot_area"),
    pluto["LotFront"].alias("lot_front"),
    pluto["LotDepth"].alias("lot_depth"),
    tax_latest["LOT_IRREG"].alias("lot_irreg"),

    # Geospatial
    pluto["Latitude"].cast("double").alias("latitude"),
    pluto["Longitude"].cast("double").alias("longitude"),
    col("geometry_wkt").alias("polygon_wkt"),

    # H3 (generated)
    to_h3(
        pluto["Latitude"].cast("string"), 
        pluto["Longitude"].cast("string")
    ).alias("h3_index"),

    # SCD columns
    current_date().alias("effective_from"),
    lit(None).cast("date").alias("effective_to"),
    lit(True).alias("is_current"),
    lit(None).cast("string").alias("parent_site_id")  # lineage
)

In [0]:

# Write to Silver ─
dim_site.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.silver.dim_site")

print(f"dim_site rows: {dim_site.count()}") 

In [0]:
# Build base building record (still _1 at this point) ─
dim_building_base = tax_latest.select(
    col("PARID").alias("site_id"),
    col("BLDG_CLASS").alias("bldg_class"),
    to_num("BLD_FRT").alias("bld_front"),
    to_num("BLD_DEP").alias("bld_depth"),
    col("BLD_EXT").alias("bld_ext"),
    to_stories("BLD_STORY").alias("num_stories"),
    to_int("NUM_BLDGS").alias("num_bldgs"),
    to_int("YRBUILT").alias("year_built"),
    col("YRBUILT_FLAG").alias("year_built_flag"),
    to_int("YRALT1").alias("year_alt1"),
    to_int("YRALT2").alias("year_alt2"),
    to_num("GROSS_SQFT").alias("gross_sqft"),
    to_num("RESIDENTIAL_AREA_GROSS").alias("residential_sqft"),
    to_num("OFFICE_AREA_GROSS").alias("office_sqft"),
    to_num("RETAIL_AREA_GROSS").alias("retail_sqft"),
    to_num("LOFT_AREA_GROSS").alias("loft_sqft"),
    to_num("FACTORY_AREA_GROSS").alias("factory_sqft"),
    to_num("WAREHOUSE_AREA_GROSS").alias("warehouse_sqft"),
    to_num("STORAGE_AREA_GROSS").alias("storage_sqft"),
    to_num("GARAGE_AREA").alias("garage_sqft"),
    to_num("OTHER_AREA_GROSS").alias("other_sqft"),
    to_int("UNITS").alias("num_units"),
    to_int("COOP_APTS").alias("coop_apts"),
    col("CONDO_Number").alias("condo_number"),
    col("COOP_NUM").alias("coop_num"),
    col("APTNO").alias("apt_no"),
    col("RECTYPE").alias("rec_type"),
    col("NOAV").alias("building_in_progress"),
    current_date().alias("effective_from"),
    lit(None).cast("date").alias("effective_to"),
    lit(True).alias("is_current")
)

#  Explode one row per building 
dim_building_expanded = (
    dim_building_base
    .withColumn(
        "n",
        when(col("num_bldgs").isNull() | (col("num_bldgs") < 1), lit(1))
        .otherwise(col("num_bldgs"))
    )
    .withColumn("bldg_seq", explode(sequence(lit(1), col("n"))))
    # building_id = PARID_1, PARID_2, etc.
    .withColumn("building_id", concat(col("site_id"), lit("_"), col("bldg_seq").cast("string")))
    .withColumn(
        "data_quality",
        when(col("n") > 1, lit("SYNTHETIC_SPLIT")).otherwise(lit("SOURCE"))
    )
    # Divide area metrics equally — flagged above for transparency
    .withColumn("gross_sqft",       col("gross_sqft")       / col("n"))
    .withColumn("residential_sqft", col("residential_sqft") / col("n"))
    .withColumn("office_sqft",      col("office_sqft")      / col("n"))
    .withColumn("retail_sqft",      col("retail_sqft")      / col("n"))
    .withColumn("loft_sqft",        col("loft_sqft")        / col("n"))
    .withColumn("factory_sqft",     col("factory_sqft")     / col("n"))
    .withColumn("warehouse_sqft",   col("warehouse_sqft")   / col("n"))
    .withColumn("storage_sqft",     col("storage_sqft")     / col("n"))
    .withColumn("garage_sqft",      col("garage_sqft")      / col("n"))
    .withColumn("other_sqft",       col("other_sqft")       / col("n"))
    .withColumn("num_units",        (col("num_units")       / col("n")).cast("int"))
    .withColumn("coop_apts",        (col("coop_apts")       / col("n")).cast("int"))
    .drop("n", "bldg_seq")
    # Reorder so building_id is first
    .select(
        "building_id", "site_id", "bldg_class", "bld_front", "bld_depth",
        "bld_ext", "num_stories", "num_bldgs", "year_built", "year_built_flag",
        "year_alt1", "year_alt2", "gross_sqft", "residential_sqft", "office_sqft",
        "retail_sqft", "loft_sqft", "factory_sqft", "warehouse_sqft", "storage_sqft",
        "garage_sqft", "other_sqft", "num_units", "coop_apts", "condo_number",
        "coop_num", "apt_no", "rec_type", "building_in_progress",
        "data_quality", "effective_from", "effective_to", "is_current"
    )
)

In [0]:
# Sanity check 
total       = dim_building_expanded.count()
source_rows = dim_building_expanded.filter(col("data_quality") == "SOURCE").count()
split_rows  = dim_building_expanded.filter(col("data_quality") == "SYNTHETIC_SPLIT").count()

print(f"Total rows after expansion : {total:,}")
print(f"  SOURCE rows              : {source_rows:,}")
print(f"  SYNTHETIC_SPLIT rows     : {split_rows:,}")

# Spot check the example parcel
dim_building_expanded \
    .filter(col("site_id") == "1000157501") \
    .select("building_id", "site_id", "num_bldgs", "num_stories",
            "year_built", "gross_sqft", "data_quality") \
    .show(truncate=False)


In [0]:
# Write to Silver ─
dim_building_expanded.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.silver.dim_building")

print("dim_building written with per-building rows")

In [0]:
# Generate floors from num_stories ─
# For each building, create one row per floor
dim_floor = dim_building_expanded \
    .filter(
        (col("num_stories").isNotNull()) & 
        (col("num_stories") > 0)
    ) \
    .select(
        col("building_id"),
        col("site_id"),
        col("num_stories")
    ) \
    .withColumn(
        "floor_number",
        explode(sequence(lit(1), col("num_stories")))
    ) \
    .select(
        # floor_id = building_id + floor number
        concat(
            col("building_id"), lit("_F"), col("floor_number")
        ).alias("floor_id"),
        col("building_id"),
        col("site_id"),
        col("floor_number"),
        # floor type
        when(col("floor_number") == 1, "ground")
        .when(col("floor_number") == col("num_stories"), "top")
        .otherwise("upper")
        .alias("floor_type")
    )

In [0]:
#  Write to Silver ─
dim_floor.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.silver.dim_floor")

print(f"dim_floor rows: {dim_floor.count()}")

In [0]:
# Build dim_owner ─
# Get unique owners with their first and last seen dates
dim_owner = tax \
    .filter(col("OWNER").isNotNull()) \
    .withColumn("owner_name_clean", 
        upper(trim(col("OWNER")))
    ) \
    .groupBy("owner_name_clean") \
    .agg(
        spark_min("YEAR").cast("int").alias("first_seen_year"),
        spark_max("YEAR").cast("int").alias("last_seen_year")
    ) \
    .withColumn(
        # Generate stable owner_id from name hash
        "owner_id", 
        sha2(col("owner_name_clean"), 256)
    ) \
    .select(
        col("owner_id"),
        col("owner_name_clean").alias("owner_name"),
        col("first_seen_year"),
        col("last_seen_year"),
        current_date().alias("effective_from"),
        lit(None).cast("date").alias("effective_to"),
        lit(True).alias("is_current")
    )

In [0]:
# Write to Silver ─
dim_owner.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.silver.dim_owner")

print(f"dim_owner rows: {dim_owner.count()}")

In [0]:
fact_tax = (
    tax
    .withColumn(
        "assessment_id",
        sha2(
            concat(
                col("PARID"), lit("_"),
                col("YEAR").cast("string"), lit("_"),
                col("PERIOD").cast("string")
            ),
            256
        )
    )
    .withColumn("owner_id", sha2(upper(trim(col("OWNER"))), 256))
    .select(

        #  Keys ─
        col("assessment_id"),
        col("PARID").alias("site_id"),
        # building_id NULL — assessment is at parcel level in source.
        # Cannot determine which specific building it applies to.
        lit(None).cast("string").alias("building_id"),
        col("owner_id"),
        col("YEAR").cast("int").alias("tax_year"),
        col("PERIOD"),

        #  Classification ─
        # Tax class exists per snapshot — use CUR as primary, fall back to FIN
        coalesce(col("CURTAXCLASS"), col("FINTAXCLASS")).alias("tax_class"),
        col("PYTAXCLASS").alias("tax_class_prior_year"),
        col("TENTAXCLASS").alias("tax_class_tentative"),
        col("FINTAXCLASS").alias("tax_class_final"),
        col("BLDG_CLASS").alias("bldg_class"),
        col("ZONING").alias("zoning"),
        col("ROLL_SECTION").alias("roll_section"),

        #  Prior Year (PY) 
        to_num("PYMKTLAND").alias("py_mkt_land"),
        to_num("PYMKTTOT").alias("py_mkt_total"),
        to_num("PYACTLAND").alias("py_act_land"),
        to_num("PYACTTOT").alias("py_act_total"),
        to_num("PYTRNLAND").alias("py_trn_land"),
        to_num("PYTRNTOT").alias("py_trn_total"),
        to_num("PYTXBTOT").alias("py_txb_total"),

        #  Tentative (TEN) 
        to_num("TENMKTLAND").alias("ten_mkt_land"),
        to_num("TENMKTTOT").alias("ten_mkt_total"),
        to_num("TENACTLAND").alias("ten_act_land"),
        to_num("TENACTTOT").alias("ten_act_total"),
        to_num("TENTRNLAND").alias("ten_trn_land"),
        to_num("TENTRNTOT").alias("ten_trn_total"),
        to_num("TENTXBTOT").alias("ten_txb_total"),

        #  Combined (CBN) ─
        to_num("CBNMKTLAND").alias("cbn_mkt_land"),
        to_num("CBNMKTTOT").alias("cbn_mkt_total"),
        to_num("CBNACTLAND").alias("cbn_act_land"),
        to_num("CBNACTTOT").alias("cbn_act_total"),
        to_num("CBNTRNLAND").alias("cbn_trn_land"),
        to_num("CBNTRNTOT").alias("cbn_trn_total"),
        to_num("CBNTXBTOT").alias("cbn_txb_total"),

        #  Final (FIN) 
        to_num("FINMKTLAND").alias("fin_mkt_land"),
        to_num("FINMKTTOT").alias("fin_mkt_total"),
        to_num("FINACTLAND").alias("fin_act_land"),
        to_num("FINACTTOT").alias("fin_act_total"),
        to_num("FINTRNLAND").alias("fin_trn_land"),
        to_num("FINTRNTOT").alias("fin_trn_total"),
        to_num("FINTXBTOT").alias("fin_txb_total"),

        #  Current (CUR) 
        to_num("CURMKTLAND").alias("cur_mkt_land"),
        to_num("CURMKTTOT").alias("cur_mkt_total"),
        to_num("CURACTLAND").alias("cur_act_land"),
        to_num("CURACTTOT").alias("cur_act_total"),
        to_num("CURTRNLAND").alias("cur_trn_land"),
        to_num("CURTRNTOT").alias("cur_trn_total"),
        to_num("CURTXBTOT").alias("cur_txb_total"),

        #  Building metrics at time of filing ─
        to_num("GROSS_SQFT").alias("gross_sqft"),
        to_num("RESIDENTIAL_AREA_GROSS").alias("residential_sqft"),
        to_num("OFFICE_AREA_GROSS").alias("office_sqft"),
        to_num("RETAIL_AREA_GROSS").alias("retail_sqft"),
        to_num("HOTEL_AREA_GROSS").alias("hotel_sqft"),
        to_num("LOFT_AREA_GROSS").alias("loft_sqft"),
        to_num("FACTORY_AREA_GROSS").alias("factory_sqft"),
        to_num("WAREHOUSE_AREA_GROSS").alias("warehouse_sqft"),
        to_num("STORAGE_AREA_GROSS").alias("storage_sqft"),
        to_num("GARAGE_AREA").alias("garage_sqft"),
        to_num("OTHER_AREA_GROSS").alias("other_sqft"),
        to_int("BLD_STORY").alias("num_stories"),
        to_int("NUM_BLDGS").alias("num_bldgs"),
        to_int("UNITS").alias("num_units"),
        to_int("YRBUILT").alias("year_built"),

        #  Flags 
        col("NOAV").alias("building_in_progress"),
        col("NEWDROP").alias("new_drop_flag"),
        col("PYTAXFLAG").alias("py_tax_flag"),
        col("TENTAXFLAG").alias("ten_tax_flag"),
        col("FINTAXFLAG").alias("fin_tax_flag"),
        col("CURTAXFLAG").alias("cur_tax_flag"),

        #  Metadata ─
        current_date().alias("load_date")
    )
)

#  Sanity checks ─
print(f"Total rows             : {fact_tax.count():,}")
print(f"Distinct PARIDs        : {fact_tax.select('site_id').distinct().count():,}")
print(f"Rows with cur_act_total: {fact_tax.filter(col('cur_act_total').isNotNull()).count():,}")

fact_tax.groupBy("tax_year").count().orderBy("tax_year").show(30)

In [0]:
# Write to Silver ─
fact_tax.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.silver.fact_tax_assessment")

print("fact_tax_assessment written to Silver")

In [0]:
# Deduplicate to one record per PARID + YEAR 
# Use latest PERIOD within each year so we compare stable end-of-year owner
w_dedup = Window.partitionBy("PARID", "YEAR").orderBy(col("PERIOD").desc())

tax_annual = (
    tax
    .filter(col("OWNER").isNotNull())
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
    .select(
        col("PARID"),
        col("YEAR").cast("int").alias("year"),
        upper(trim(col("OWNER"))).alias("owner_name_clean"),
        # Carry forward useful context for the sale record
        coalesce(col("CURTAXCLASS"), col("FINTAXCLASS")).alias("tax_class"),
        col("BLDG_CLASS").alias("bldg_class"),
        col("ZONING").alias("zoning")
    )
)

# Detect owner changes with LAG ─
w_parcel = Window.partitionBy("PARID").orderBy("year")

sales_raw = (
    tax_annual
    .withColumn("prev_owner", lag("owner_name_clean").over(w_parcel))
    .withColumn("prev_year",  lag("year").over(w_parcel))
    # Only keep rows where owner actually changed
    .filter(
        col("prev_owner").isNotNull() &
        (col("owner_name_clean") != col("prev_owner"))
    )
)

fact_sale_event = (
    sales_raw
    .withColumn(
        "sale_event_id",
        sha2(
            concat(
                col("PARID"), lit("_"),
                col("year").cast("string")
            ),
            256
        )
    )
    .withColumn("seller_id", sha2(col("prev_owner"), 256))
    .withColumn("buyer_id",  sha2(col("owner_name_clean"), 256))
    .select(
            col("sale_event_id"),                               # PK
            col("PARID").alias("site_id"),                     # FK → dim_site
            # building_id intentionally NULL — sale is recorded at parcel level
            # in source data. Cannot determine which specific building transferred.
            # Populate from ACRIS when available.
            lit(None).cast("string").alias("building_id"),
            col("year").alias("sale_year"),
            col("prev_year").alias("prior_ownership_year"),
            # Seller
            col("seller_id"),                                  # FK → dim_owner
            col("prev_owner").alias("seller_name"),
            # Buyer
            col("buyer_id"),                                   # FK → dim_owner
            col("owner_name_clean").alias("buyer_name"),
            # Property context at time of transfer
            col("tax_class"),
            col("bldg_class"),
            col("zoning"),

            # Metadata
            lit("INFERRED").alias("source_type"),
            current_date().alias("load_date")
        )
    )

# COMMAND ----------

# Sanity checks ─
total      = fact_sale_event.count()
distinct   = fact_sale_event.select("site_id").distinct().count()

print(f"Total inferred sale events : {total:,}")
print(f"Distinct parcels with sales: {distinct:,}")

# Year distribution of transfers
fact_sale_event.groupBy("sale_year") \
    .count() \
    .orderBy("sale_year") \
    .show(30)

In [0]:
fact_sale_event.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.silver.fact_sale_event")

print("fact_sale_event written to Silver")